# survjax playground

An editable companion to the [SurvivalData guide](survival-data.md),
[distribution pages](builtin-families.md) and [concordance guide](concordance.md).
All data are synthetic: the observation examples follow six patients in
months since diagnosis, with progression as the event. A separate four-patient
cohort below illustrates risk ranking. Exact/right likelihoods without
truncation are also available; see [log likelihoods](likelihood.md).

Open this notebook with `uv run --locked --group docs jupyter lab
docs/playground.ipynb` from the repository root. Change values and rerun cells.
The assertions check the defaults: if you edit scores or observations,
recalculate the expected counts and fractions too. The last section lists
things to try.


In [ ]:
import jax
import jax.numpy as jnp

import survjax

nan, inf = jnp.nan, jnp.inf
print(f"JAX float64 enabled: {jax.config.jax_enable_x64}")
data = survjax.SurvivalData(jnp.array([4.0, 12.0, 7.0, 9.0]), event=[1, 0, 1, 0])

## Time and status

Patients 1 and 3 progressed at months 4 and 7 (`event=1`). Patients 2 and 4
were last seen without progression at months 12 and 9 (`event=0`), so they are
right censored. The printout shows an exact time for events and the set of
possible event times, `(lower, upper]`, for censored rows.

In [ ]:
data

The printout is built from five stored arrays with one entry per patient.
`exact` marks events, `lower` and `upper` bound the event time, and the
truncation window `(left_trunc, right_trunc]` defaults to `(0, inf]`.

In [ ]:
print("exact:      ", data.exact)
print("lower:      ", data.lower)
print("upper:      ", data.upper)
print("left_trunc: ", data.left_trunc)
print("right_trunc:", data.right_trunc)

## The full cohort

Two more patients joined two months after diagnosis, so both are left
truncated at month 2. Patient 5 had a clear scan at month 6 and progression at
month 9 (interval censored). Patient 6 already showed progression at the first
scan at month 3 (left censored). Keyword flags describe each kind; a row
with no flag is an exact event, and `time2` is read only on interval rows.

In [ ]:
cohort = survjax.SurvivalData(
    [4.0, 12.0, 7.0, 9.0, 6.0, 3.0],
    right_censored=[False, True, False, True, False, False],
    left_censored=[False, False, False, False, False, True],
    interval_censored=[False, False, False, False, True, False],
    time2=[nan, nan, nan, nan, 9.0, nan],
    left_trunc=[0.0, 0.0, 0.0, 0.0, 2.0, 2.0],
)
cohort

## The same cohort as endpoints

`from_bounds` takes each patient's set directly. Exact events need the
`exact` mask; their upper endpoint may be `nan`. The printout should match the
one above.

In [ ]:
same_cohort = survjax.SurvivalData.from_bounds(
    [4.0, 12.0, 7.0, 9.0, 6.0, 0.0],
    [nan, inf, nan, inf, 9.0, 3.0],
    exact=[True, False, True, False, False, False],
    left_trunc=[0.0, 0.0, 0.0, 0.0, 2.0, 2.0],
)
same_cohort

## Checking data

Construction never stops on bad values. It returns the data, notes problems
in a footer, and records them in `data.issues`. These five records each have
a problem: an exact event at month 0, an interval with no end, an interval
that ends before it starts, a patient censored before entering the study, and
an interval `(0, inf]` that rules nothing out (valid, but uninformative).

In [ ]:
bad = survjax.SurvivalData(
    [0.0, 6.0, 9.0, 1.0, 0.0],
    right_censored=[False, False, False, True, False],
    interval_censored=[False, True, True, False, True],
    time2=[nan, nan, 6.0, nan, inf],
    left_trunc=[0.0, 0.0, 0.0, 2.0, 0.0],
)
bad

In [ ]:
bad.issues

Each rule is also a boolean mask with one entry per row, and `ok` is one
boolean for the whole data set:

In [ ]:
print("invalid_rows: ", bad.issues.invalid_rows)
print("invalid_time: ", bad.issues.invalid_time)
print("uninformative:", bad.issues.uninformative)
print("ok:           ", bad.issues.ok)

To stop a script on bad data, call `raise_if_invalid()`. Here the error is
caught and printed so that the notebook keeps running:

In [ ]:
try:
    bad.issues.raise_if_invalid()
except survjax.InvalidDataError as error:
    print(error)

## Inside `jax.jit`

Construction and `data.issues` work in compiled code. Return the issues and
check them afterwards. Printing inside the compiled function shows only the
shape, because the values are not known there yet.

In [ ]:
@jax.jit
def diagnose(time, status):
    data = survjax.SurvivalData(time, status)
    print("traced:", data)
    return data.issues


diagnose(jnp.array([4.0, 12.0, 7.0]), jnp.array([1, 0, 2]))

## A Weibull model for progression

Suppose time to progression follows a Weibull distribution with shape 1.5
(a rising hazard) and scale 12 months, so that 37% of patients are still
progression free at month 12. Evaluate it at the four patients' times:

In [ ]:
model = survjax.Weibull(shape=1.5, scale=12.0)
months = jnp.array([4.0, 12.0, 7.0, 9.0])
print("survival:  ", model.survival(months))
print("hazard:    ", model.hazard(months))
print("log density:", model.log_prob(months))

Two arms form a batch of two distributions. `survival_grid` evaluates both on
one shared vector of times: one row per time, one column per arm.

In [ ]:
arms = survjax.Weibull(shape=1.5, scale=jnp.array([12.0, 18.0]))
grid = jnp.array([0.0, 6.0, 12.0, 24.0])
arms.survival_grid(grid)

Parameters can also come from log values, for example the output of an
optimiser, and gradients flow through them:

In [ ]:
def survival_at_year(log_scale):
    return survjax.Weibull.from_log(
        log_shape=jnp.log(1.5), log_scale=log_scale
    ).survival(12.0)


survival_at_year(jnp.log(12.0)), jax.grad(survival_at_year)(jnp.log(12.0))

## Rank predictions with Harrell's C

Supply one finite score per patient, with higher values predicting earlier
events. survjax does not choose a score summary or fit a model. Concordance
supports exact events and right censoring without truncation; no useful
score gradients are promised. The observation axis N is last.

Use the [guide's four-patient example](concordance.md): pairs (1,2), (1,3)
and (2,3) are concordant; (1,4) has tied scores; (2,4) is discordant. Pair
(2,3) compares an event with censoring at the same time, event first.
Pair (3,4) is excluded because censoring does not order the event times.
Thus C = (3 + 1/2) / 5 = 7/10.


In [ ]:
rank_time = jnp.array([1.0, 2.0, 2.0, 3.0])
rank_event = jnp.array([True, True, False, True])
rank_scores = jnp.array([4.0, 3.0, 1.0, 4.0])
rank_data = survjax.SurvivalData(rank_time, rank_event)
rank_c, rank_info = survjax.concordance_index_with_info(rank_scores, rank_data)
rank_counts = tuple(
    int(count.hi) * 2**32 + int(count.lo)
    for count in (
        rank_info.concordant,
        rank_info.discordant,
        rank_info.tied_score,
        rank_info.comparable,
    )
)
assert rank_counts == (3, 1, 1, 5)  # Hand-enumerated above.
assert bool(rank_c == jnp.asarray(7 / 10, dtype=rank_c.dtype))
print("C:", rank_c, "counts (concordant, discordant, tied, comparable):", rank_counts)

### Score draws and compiled evaluation

Each row below supplies another score vector for the same patients, shape
(D,N) = (3,4), giving three separate C values. Reverse the scores to swap
concordant and discordant pairs: C = (1 + 1/2) / 5 = 3/10. Constant scores
get half credit for all five pairs: C = 1/2. The metric never averages draws.
The same calls work inside `jit` and when mapped over score rows with `vmap`.


In [ ]:
score_draws = jnp.stack([rank_scores, -rank_scores, jnp.ones_like(rank_scores)])
draw_c = survjax.concordance_index(score_draws, rank_data)
compiled_c = jax.jit(survjax.concordance_index)(score_draws, rank_data)
mapped_c = jax.vmap(survjax.concordance_index, in_axes=(0, None))(score_draws, rank_data)
expected_c = jnp.asarray([7 / 10, 3 / 10, 1 / 2], dtype=draw_c.dtype)
assert draw_c.shape == (3,)
assert bool(jnp.array_equal(draw_c, expected_c))  # The three fractions above.
assert bool(jnp.array_equal(compiled_c, expected_c))
assert bool(jnp.array_equal(mapped_c, expected_c))
print("eager:", draw_c, "jit:", compiled_c, "vmap:", mapped_c)

### Check whether a result can be evaluated

The earlier six-patient `cohort` includes interval/left censoring and delayed
entry. Its observations are valid, but the whole concordance result is
unsupported: NaN, zero counts, `unsupported=True`. Rows are never dropped.
A cohort with all patients right censored also gives NaN, but with
`no_comparable_pairs=True` instead. Invalid observations or nonfinite scores
set `invalid`; that mask may overlap `unsupported`. These masks describe
whole results rather than individual patients.


In [ ]:
unsupported_c, modes = survjax.concordance_index_with_info(
    jnp.ones(cohort.shape), cohort
)
assert bool(cohort.issues.ok)
assert bool(jnp.isnan(unsupported_c)) and bool(modes.unsupported)
assert not bool(modes.invalid | modes.no_comparable_pairs)
assert int(modes.comparable.hi) == int(modes.comparable.lo) == 0
undefined_c, no_pairs = survjax.concordance_index_with_info(
    rank_scores, time=rank_time, event=jnp.zeros_like(rank_event)
)
assert bool(jnp.isnan(undefined_c)) and bool(no_pairs.no_comparable_pairs)
assert not bool(no_pairs.invalid | no_pairs.unsupported)
print("mixed cohort:", unsupported_c, "unsupported:", modes.unsupported)
print("all censored:", undefined_c, "no pairs:", no_pairs.no_comparable_pairs)

## Things to try

- Give patient 1 status `2`, as in R's 1/2 coding. Which rule fires?
- In `from_bounds`, set patient 1's upper endpoint to `4.0` and drop
  `exact=True` for that row. Equal endpoints alone are not an event.
- Move patient 5's entry (`left_trunc`) to month 10, after its interval.
- Add `right_trunc=24.0` to the cohort and look at the window column.
- Set the Weibull shape to 1 and compare with `survjax.Exponential(rate=1 / 12)`.
- Pass the grid to `arms.survival(grid)` instead of `survival_grid`. Why does
  it fail, and what happens with a grid of two times?
- Evaluate `model.log_prob(0.0)` for shapes 0.5, 1 and 2.
- Restart JupyterLab with `JAX_ENABLE_X64=1` in the environment to see float64.

- Change a concordance score, then enumerate the five pairs again. Which
  counts change? Update the expected counts and fractions before rerunning.
- Set all concordance scores equal, or negate the scores, and check the result.
- Set a score to NaN using `.at[0].set(jnp.nan)` and inspect `invalid`.
- Make all `rank_event` entries false and inspect `no_comparable_pairs`.
